# Model selection code based on HW5
### Found SVC with C= 1 and rbf to be the most accurate model at 0.88 on validation data (80/10/10 split)

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn import svm, linear_model, datasets
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import (confusion_matrix, precision_score, recall_score,
                             accuracy_score, roc_auc_score, RocCurveDisplay, ConfusionMatrixDisplay)

from sklearn.datasets import make_classification
from sklearn.ensemble import GradientBoostingClassifier
from imblearn.over_sampling import RandomOverSampler

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer



In [2]:
df = pd.read_csv("heart.zip")
def resumetable(df):
    print(f"data shape: {df.shape}")
    summary = pd.DataFrame(df.dtypes, columns = ['data type'])
    summary = summary.reset_index()
    summary = summary.rename(columns = {"index": "feature"})
    summary["Num_Null"] = df.isnull().sum().values
    summary["Num_Unique"] = df.nunique().values
    summary["First_Value"] = df.loc[0].values
    summary["Second_Value"] = df.loc[1].values
    summary["Third_Value"] = df.loc[2].values
    
    return summary
resumetable(df)

data shape: (918, 12)


,feature,data type,Num_Null,Num_Unique,First_Value,Second_Value,Third_Value
0,Age,int64,0,50,40,49,37
1,Sex,object,0,2,M,F,M
2,ChestPainType,object,0,4,ATA,NAP,ATA
3,RestingBP,int64,0,67,140,160,130
4,Cholesterol,int64,0,222,289,180,283
5,FastingBS,int64,0,2,0,0,0
6,RestingECG,object,0,3,Normal,Normal,ST
7,MaxHR,int64,0,119,172,156,98
8,ExerciseAngina,object,0,2,N,N,N
9,Oldpeak,float64,0,53,0.0,1.0,0.0


In [3]:
df

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,40,M,ATA,140,289,0,Normal,172,N,0.0,Up,0
1,49,F,NAP,160,180,0,Normal,156,N,1.0,Flat,1
2,37,M,ATA,130,283,0,ST,98,N,0.0,Up,0
3,48,F,ASY,138,214,0,Normal,108,Y,1.5,Flat,1
4,54,M,NAP,150,195,0,Normal,122,N,0.0,Up,0
...,...,...,...,...,...,...,...,...,...,...,...,...
913,45,M,TA,110,264,0,Normal,132,N,1.2,Flat,1
914,68,M,ASY,144,193,1,Normal,141,N,3.4,Flat,1
915,57,M,ASY,130,131,0,Normal,115,Y,1.2,Flat,1
916,57,F,ATA,130,236,0,LVH,174,N,0.0,Flat,1


In [4]:
X = df.drop(['HeartDisease'], axis = 1)
y = df['HeartDisease']

numeric_features = ["Age", "RestingBP", "Cholesterol", "MaxHR", "Oldpeak"]
categorical_features = ["Sex", "ChestPainType", "FastingBS", "RestingECG", "ExerciseAngina", "ST_Slope"]


In [5]:
# split 80% training data, 20% "_tmp" for validation & test
X_train, X_tmp, y_train, y_tmp = \
    train_test_split(X, y, test_size=.2,random_state=0, stratify=y)
# of remaining 20%, split in half to get 10% validation, 10% test
X_valid, X_test, y_valid, y_test = \
    train_test_split(X_tmp, y_tmp, test_size=.5,random_state=0, stratify=y_tmp)

In [6]:
# Identify features
numeric_features = ["Age", "RestingBP", "Cholesterol", "MaxHR", "Oldpeak"]
categorical_features = ["Sex", "ChestPainType", "FastingBS", "RestingECG", "ExerciseAngina", "ST_Slope"]

# Preprocessing steps
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])


In [7]:
models = [svm.SVC(), LogisticRegression(max_iter=5000), 
          DecisionTreeClassifier(criterion='entropy'), KNeighborsClassifier()]

params = [
    {'classifier__kernel': ['linear', 'rbf'], 'classifier__C': [0.01, 1, 100]},
    {'classifier__C': [0.01, 1, 100]},
    {'classifier__max_depth': [1, 3, 5, 7]},
    {'classifier__n_neighbors': [1, 2, 3, 4]}
]

best_score = -np.inf
best_clf_index = -1
best_clf = None

for i, model in enumerate(models):
    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", model)
    ])
    
    grid = GridSearchCV(pipe, param_grid=params[i], cv=5)
    grid.fit(X_train, y_train)

    val_score = grid.score(X_valid, y_valid)
    
    print(f"Classifier: {model.__class__.__name__}")
    print(f"Best parameters: {grid.best_params_}")
    print(f"Validation accuracy: {val_score:.3f}\n")
    
    if val_score > best_score:
        best_score = val_score
        best_clf_index = i
        best_clf = grid.best_estimator_


Classifier: SVC
Best parameters: {'classifier__C': 1, 'classifier__kernel': 'rbf'}
Validation accuracy: 0.880

Classifier: LogisticRegression
Best parameters: {'classifier__C': 1}
Validation accuracy: 0.870

Classifier: DecisionTreeClassifier
Best parameters: {'classifier__max_depth': 5}
Validation accuracy: 0.826

Classifier: KNeighborsClassifier
Best parameters: {'classifier__n_neighbors': 3}
Validation accuracy: 0.837

